In [1]:
import malariagen_data

import dask
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from Bio import bgzf

ag3 = malariagen_data.Ag3()

ag3

# Useful tutorial for what follows
# https://anopheles-genomic-surveillance.github.io/workshop-3/module-3-snps.html

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : None
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.10.0
Client location                       : Alberta, Canada
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.10.0/Ag3.html

In [6]:
def filter_and_export_vcf_bgz(ds, out_path, sample_name_map):
    contigs = np.asarray(ds.attrs["contigs"])
    raw_samples = ds["sample_id"].values.astype(str)

    # Replace VBS... IDs with Lucas's sample.name for the VCF header
    samples = [sample_name_map[s] for s in raw_samples]

    n_variants = ds.sizes["variants"]
    variant_chunks = ds["call_genotype"].chunks[0]

    n_written = 0
    with bgzf.open(out_path, "wt") as f, tqdm(
        total=n_variants,
        desc="Writing BGZF VCF",
        unit="variants",
        unit_scale=True,
    ) as pbar:
        f.write("##fileformat=VCFv4.2\n")
        for c in contigs:
            f.write(f"##contig=<ID={c}>\n")
        f.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
        header_cols = ["#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"] + samples
        f.write("\t".join(header_cols) + "\n")

        start = 0
        for chunk_len in variant_chunks:
            end = start + chunk_len

            c_idx, pos, alleles, gt = dask.compute(
                ds["variant_contig"].data[start:end],
                ds["variant_position"].data[start:end],
                ds["variant_allele"].data[start:end],
                ds["call_genotype"].data[start:end],
            )

            no_missing = (gt >= 0).all(axis=(1, 2))
            is_segregating = (gt != gt[:, :1, :1]).any(axis=(1, 2))
            keep = no_missing & is_segregating

            if np.any(keep):
                c_sub = contigs[c_idx[keep]]
                p_sub = pos[keep]
                a_sub = alleles[keep].astype("U1")
                g_sub = gt[keep]

                lines = []
                for i in range(len(p_sub)):
                    obs_alts = [a for a in (1, 2, 3) if np.any(g_sub[i] == a)]
                    ref = a_sub[i, 0]
                    alt_str = ",".join(a_sub[i, a] for a in obs_alts)

                    remapped = g_sub[i].copy()
                    for new_idx, old_idx in enumerate(obs_alts, start=1):
                        remapped[g_sub[i] == old_idx] = new_idx

                    a1 = remapped[:, 0].astype(str)
                    a2 = remapped[:, 1].astype(str)
                    gt_strs = np.char.add(np.char.add(a1, "/"), a2)

                    row = (
                        f"{c_sub[i]}\t{p_sub[i]}\t.\t{ref}\t{alt_str}\t.\tPASS\t.\tGT\t"
                        + "\t".join(gt_strs)
                    )
                    lines.append(row)

                f.write("\n".join(lines) + "\n")
                n_written += len(lines)

            pbar.update(chunk_len)
            start = end

    print(f"Done. Wrote {n_written:,} variants for {len(samples)} samples to {out_path}.")

In [9]:
# From https://github.com/LSTM-VIGG/GAARD_work/blob/main/NGSrelate/full_relatedness/sib_group_table.csv
gaard_df = pd.read_csv("sib_group_table.tsv", sep="\t")
partner_ids = set(gaard_df["sample.name"])

meta = ag3.sample_metadata()
matched_meta = meta[meta["partner_sample_id"].isin(partner_ids)].drop_duplicates(subset=["sample_id"])

print(f"Matched {len(matched_meta)} / {len(partner_ids)} IDs across {matched_meta['sample_set'].nunique()} sample sets.")
print(matched_meta["sample_set"].value_counts())

id_to_sample_name = dict(zip(matched_meta["sample_id"], matched_meta["partner_sample_id"]))

Matched 238 / 238 IDs across 4 sample sets.
sample_set
1244-VO-GH-YAWSON-VMF00051       160
1253-VO-TG-DJOGBENOU-VMF00052     58
1237-VO-BJ-DJOGBENOU-VMF00050     18
1245-VO-CI-CONSTANT-VMF00054       2
Name: count, dtype: int64


In [10]:
samples = matched_meta["sample_set"].unique().tolist()

len_2L = ag3.genome_sequence("2L").shape[0]
len_2R = ag3.genome_sequence("2R").shape[0]

# 2L and 2R have regions that need to be removed
# https://github.com/LSTM-VIGG/GAARD_work/blob/c97d60e932e454c06b75c07f73cc5ce6e4af50a6/focal_gwas/Fst/focal_gwas_Fst.r#L15
contig_regions = {
    #"2L": ["2L:1-20523999", f"2L:42165601-{len_2L}"],
    "2R": ["2R:1-3099999", f"2R:3950001-{len_2R}"],
    "3L": ["3L"],
    "3R": ["3R"]
}

for (contig, region) in contig_regions.items():
    ds_snps = ag3.snp_calls(region=region, sample_sets=samples, site_mask="gamb_colu_arab")
    sample_mask = np.isin(ds_snps["sample_id"].values, matched_meta["sample_id"].values)
    ds_subset = ds_snps.isel(samples=sample_mask)
    filter_and_export_vcf_bgz(ds_subset, f"GAARD-{contig}.vcf.gz", id_to_sample_name)

Writing BGZF VCF:   0%|          | 0.00/39.9M [00:00<?, ?variants/s]

Done. Wrote 6,780,492 variants for 238 samples to GAARD-2R.vcf.gz.
                                   

Writing BGZF VCF:   0%|          | 0.00/25.9M [00:00<?, ?variants/s]

Done. Wrote 4,861,371 variants for 238 samples to GAARD-3L.vcf.gz.
                                   

Writing BGZF VCF:   0%|          | 0.00/33.4M [00:00<?, ?variants/s]

Done. Wrote 6,626,886 variants for 238 samples to GAARD-3R.vcf.gz.
